[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_1/L4.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 1 | BÀI 4


---

# **MÔ HÌNH CÓ GIÁM SÁT: CA NGHIÊN CỨU PHÂN LOẠI**

|  |  |
|:---|:---|
|**Thời gian đọc** |  60 phút |
|**Kiến thức nền** | Hồi quy logistic, Ma trận nhầm lẫn  |
|**Từ khóa** | Xác suất vỡ nợ  |

---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_1/"
FILES = ['bank.csv']
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
        print("Đã tải:", f)

*Ở bài cuối của Module 1, ta xây dựng một ứng dụng mô hình phân loại học máy để dự báo xác suất vỡ nợ của người xin vay.*

## **1. Dự báo xác suất vỡ nợ**

Xác suất vỡ nợ cho biết khả năng một người đi vay không đáp ứng được nghĩa vụ trả nợ (lãi hoặc gốc). Vỡ nợ nghĩa là bên cho vay có quyền hợp pháp thử thu hồi khoản nợ bằng cách xử lý tài sản của người vay. Xác suất vỡ nợ của người vay càng cao thì lãi suất bên cho vay áp dụng càng cao, hoặc số tiền người vay đủ điều kiện nhận từ ngân hàng càng thấp.

Ta sẽ khai thác dữ liệu thực tế về các sự kiện vỡ nợ của một ngân hàng ở Israel. Trong bài này, ta xây dựng mô hình chấm điểm tín dụng để xác định xác suất một người vay vỡ nợ, bằng cách huấn luyện mô hình trên một phần dữ liệu. Ta huấn luyện mô hình dựa trên lịch sử các khoản vay đã cấp và đặc điểm cá nhân của người vay (như tuổi, trình độ học vấn, tỷ lệ nợ trên thu nhập và các biến khác).

Ta đọc dữ liệu.

In [ ]:
import pandas as pd

df = pd.read_csv("bank.csv")
df.head()

Tập dữ liệu gồm một mẫu vài chục nghìn khoản vay đã được cấp trước đây, với 41.188 bản ghi và 10 trường. Dữ liệu có mã định danh cá nhân và cho biết mỗi khoản vay có vỡ nợ hay không ($y=0$ là không vỡ nợ và $y=1$ là vỡ nợ), cùng các đặc điểm của người vay: tuổi, trình độ học vấn (bằng đại học, trung học, mù chữ, giáo dục cơ bản và khóa học nghề), số năm làm việc tại nơi hiện tại, số năm ở cùng một nhà, thu nhập, tỷ lệ nợ trên thu nhập, dư nợ thẻ tín dụng và các khoản nợ khác.

Ta sẽ khai thác lượng thông tin mà một số biến có thể mang lại khi dự báo hành vi vỡ nợ của cá nhân, bằng mô hình hồi quy logistic. Mô hình này sẽ hữu ích cho ngân hàng hoặc tổ chức phát hành tín dụng khi ước lượng xác suất vỡ nợ của một người xin vay với những đặc điểm nhất định. Một mô hình dự báo tốt giúp ngân hàng cung cấp dịch vụ tài chính tốt hơn và phù hợp hơn với từng khách hàng.

Biến học vấn là biến phân loại có nhiều nhóm. Để ước lượng mô hình, ta tạo các biến chỉ báo riêng (hay "biến giả") cho biết cá nhân có từng trình độ học vấn đó (giá trị 1) hay không (giá trị không).

In [ ]:
cat_vars = ["education"]
for var in cat_vars:
    cat_list = "var" + "_" + var
    cat_list = pd.get_dummies(df[var], prefix=var)
    data1 = df.join(cat_list)
    df = data1
cat_vars = ["education"]
data_vars = df.columns.values.tolist()
to_keep = [i for i in data_vars if i not in cat_vars]

In [ ]:
df_final = df[to_keep]
df_final.drop(["loan_applicant_id"], axis=1, inplace=True)
df_final.head()

Như thường lệ, ta chia mẫu thành tập huấn luyện và tập kiểm tra. Tập dữ liệu có hơn 41.000 quan sát, ta chia đôi để tạo cả hai mẫu. Vỡ nợ xảy ra ở khoảng 11% số quan sát trong dữ liệu.

In [ ]:
X, y = (
    df_final.loc[:, df_final.columns != "y"],
    df_final.loc[:, df_final.columns == "y"],
)
print(X.shape, y.shape)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=int(len(y) * 0.5), shuffle=False
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
print(df_final["y"].value_counts())
print(
    "Percentage of default: ",
    100
    * df_final["y"].value_counts()[1]
    / (df_final["y"].value_counts()[0] + df_final["y"].value_counts()[1]),
)

## **2. Tiền xử lý dữ liệu**

Trước khi huấn luyện các thuật toán ML, ta cần chắc chắn mình đã hiểu rõ thông tin đang làm việc. Có được hiểu biết ban đầu về cách dữ liệu vận hành có thể định hướng hữu ích để cải thiện hiệu suất của thuật toán.

### **2.1 Thống kê tóm tắt**

Tính các thống kê tóm tắt (còn gọi là thống kê mô tả) là việc đầu tiên hữu ích, cho ta cái nhìn ban đầu về hành vi của dữ liệu. Trung bình và trung vị cho biết giá trị đại diện của từng biến trong các quan sát. Khoảng biến thiên và phương sai (hoặc độ lệch chuẩn) cho biết độ phân tán của các biến và có thể hữu ích để nhận diện giá trị ngoại lai (sẽ nói thêm bên dưới).

Với pandas, việc lấy các thống kê mô tả chính của mọi biến rất dễ dàng, như bên dưới. Nếu xem trung bình hoặc trung vị của dữ liệu như thước đo tính đại diện của từng biến, ta thấy các cá nhân trong mẫu ở độ tuổi trung niên, gắn bó lâu với công việc, đã ở cùng một nhà khá lâu, có thu nhập tương đối cao theo chuẩn Israel, trong khi mức nợ thấp (16% thu nhập). Cuối cùng, mẫu phân bố đều về học vấn trên cả năm trình độ.

Nếu xem các đặc điểm khác của phân phối các đặc trưng đầu vào, như độ phân tán, độ lệch và độ nhọn, ta thấy những biến biến động cực đoan nhất là các biến liên quan đến số tiền nợ, cả nợ thẻ tín dụng lẫn các loại nợ khác. Tức là có một số cá nhân có mức nợ rất cao về giá trị tuyệt đối so với đa số cá nhân còn lại.

In [ ]:
X.describe()

In [ ]:
print(X.skew())

In [ ]:
print(X.kurt())

Ta trực quan hóa các khía cạnh khác nhau của phân phối từng đặc trưng đầu vào để hiểu dữ liệu tốt hơn và khám phá biến nào liên quan nhiều hơn đến vỡ nợ. Ta vẽ phân phối của từng biến trên cùng một thang đo. Vì vậy, để trực quan hóa tốt hơn, ta chuẩn hóa dữ liệu, tức với mỗi đặc trưng đầu vào, trừ đi trung bình và chia cho độ lệch chuẩn. Hãy nhớ đây là một trong những cách co giãn thang đo đã nêu ở Bài 3.

Ngoài chuẩn hóa, ta vẽ phân phối của từng biến riêng cho những quan sát có vỡ nợ và không vỡ nợ. Hãy để ý trong hình dưới đây rằng bốn biến có khác biệt đáng kể nhất trong phân phối giữa nhóm không vỡ nợ và nhóm vỡ nợ: thu nhập hộ gia đình, tỷ lệ nợ trên thu nhập, dư nợ thẻ tín dụng và các khoản nợ khác. Như thống kê mô tả ở trên đã gợi ý, mức dư nợ thẻ tín dụng và các loại nợ khác có những cá nhân với giá trị rất cực đoan.

Cũng lưu ý rằng trình độ học vấn dường như không liên quan để dự báo vỡ nợ, vì phân phối ở mọi trình độ học vấn đều gần giống nhau. Từ quan sát này, ta có thể chọn loại các biến đó khỏi mô hình vì chúng dường như cho ít khả năng dự báo so với các đặc trưng khác. Tuy nhiên, ta cũng cần nhớ rằng có thể tồn tại những tương tác phi tuyến giữa các biến giúp dự báo kết quả quan tâm, dù thoạt nhìn từng biến riêng lẻ có vẻ không liên quan. Mặc dù mô hình hồi quy logistic của ta bỏ qua tương tác giữa các biến, ta cần tính đến khả năng này ở những mô hình ML nâng cao hơn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

X_mean = X.mean()
X_std = X.std()
X_std = (X - X_mean) / X_std
X_std["y"] = y
X_std = X_std.melt(id_vars=["y"], var_name="Column", value_name="Normalized")
plt.figure(figsize=(18, 6))
ax = sns.violinplot(data=X_std, x="Column", y="Normalized", hue="y", split=True)
_ = ax.set_xticklabels(X_train.keys(), rotation=90)

Để khám phá thêm mối quan hệ giữa vỡ nợ và từng đặc trưng đầu vào, ta có thể chạy các hồi quy logistic chỉ gồm một đặc trưng, ngoài hằng số chệch. Để trực quan hóa tốt hơn và dễ so sánh, bây giờ ta co giãn các biến bằng phương pháp min-max, đưa giá trị mỗi biến về khoảng đơn vị. Lại lưu ý rằng bốn biến tạo ra phần lớn tác động.

Phân tích đến đây có thể khiến ta nghĩ rằng có thể bỏ qua phần lớn các đặc trưng. Tuy nhiên, hãy lưu ý các hồi quy ở trên rất gần với hồi quy tuyến tính, vốn bỏ qua sự hiện diện của các quan hệ phi tuyến trong dữ liệu.

In [ ]:
X_minmax = (X - X.min()) / (X.max() - X.min())
X_minmax["y"] = y
plt.figure(figsize=(18, 6))
g = 1
for colname in X:
    plt.subplot(3, 4, g)
    sns.regplot(
        x=colname,
        y="y",
        data=X_minmax,
        marker="",
        logistic=True,
        fit_reg=True,
        ci=None,
        label=colname,
    )
    plt.title(colname, fontsize="large")
    plt.xlabel("")
    g += 1
    # sns.lmplot(x=colname, y='y', data=X_minmax, markers = "", logistic = True, fit_reg=True, ci=None)
fig = plt.gcf()
fig.tight_layout(pad=1.0)
plt.show()

Một bước quan trọng khác trong tiền xử lý dữ liệu là phân tích tương quan giữa các đặc trưng. Mô hình dự báo của ta rất có thể thất bại nếu thuật toán ML được nạp nhiều đặc trưng tương quan mạnh với nhau: thuật toán tối ưu sẽ khó phân tách biến nào thực sự liên quan để dự báo, làm chậm quá trình huấn luyện.

Bên dưới là ma trận tương quan của các đặc trưng đầu vào, cho biết mức đồng biến thiên giữa từng cặp đặc trưng. Các biến tỷ lệ nợ trên thu nhập, dư nợ thẻ tín dụng và nợ khác có hệ số tương quan với nhau trên 0,5. Mặc dù những tương quan này không đáng lo ngại, ta vẫn nên kiểm tra hiệu suất của các mô hình ML loại ít nhất một trong những biến này.

In [ ]:
X.corr()

Trong thực tế, một số ứng dụng có số lượng biến dự báo lớn đến mức không thể xem những chỉ số đơn giản này cho mọi biến. Tuy vậy, vẫn nên kiểm tra tối thiểu. Để việc phân tích dễ hơn:
  * Tập trung vào một nhóm biến dự báo mà theo suy luận ban đầu sẽ liên quan hơn khi dự báo kết quả ta cần, ví dụ trong ứng dụng này thu nhập và mức nợ dường như hợp lý nhất.
  * Theo dõi các giá trị ngoại lai trong thống kê tóm tắt (sẽ nói thêm bên dưới).

### **2.2 Dữ liệu thiếu và phát hiện giá trị ngoại lai**

Có hai cách chính để xử lý dữ liệu thiếu: loại bỏ và điền khuyết (imputation). Loại bỏ là cách không giả định gì nhưng tốn kém nếu phải xóa cả một quan sát chỉ vì thiếu một giá trị đặc trưng. Điền khuyết khắc phục việc mất thông tin nhưng đòi hỏi phải đưa ra các giả định, vốn thường sai.

Khi gặp dữ liệu thiếu, ta có thể chọn thay bằng trung vị hoặc trung bình tính trên mặt cắt ngang của các tài sản. Điều này có nghĩa giá trị đặc trưng bị thiếu sẽ nằm trong khối chính của các giá trị quan sát được. Nếu thiếu nhiều giá trị, ta sẽ làm thay đổi đáng kể phân phối gốc của đặc trưng, và cần đánh giá xem có nên đưa đặc trưng đó vào dữ liệu hay không.

Trong bối cảnh chuỗi thời gian, hướng tới kiểm thử ngược (backtesting) và dự báo, cách điền khuyết đơn giản nhất là thay thông tin thiếu bằng giá trị trước đó của chuỗi thời gian. Tuy nhiên nếu muốn dự báo lợi suất, nên tránh điền khuyết từ giá trị quá khứ. Mặc định, lựa chọn tốt hơn là đặt các chỉ báo lợi suất bị thiếu bằng không, thường gần với trung bình hoặc trung vị. Dù vậy, nếu một đặc trưng có tự tương quan cao, tức tương quan giữa giá trị tương lai và quá khứ của biến gần bằng một, thì điền khuyết từ quá khứ có thể hợp lý. Nếu không, nên tránh.

Trong các bài toán chuỗi thời gian, như dự báo lợi suất tương lai, ta cũng nên tránh nội suy làm cách điền dữ liệu thiếu. Nội suy theo các kỳ thời gian nghĩa là ta đang dự báo kết quả tương lai bằng thông tin chưa có ở thời điểm dự báo. Ví dụ, nếu số liệu lợi nhuận được công bố vào tháng 4 và tháng 7, nội suy cho tháng 5 và tháng 6 đòi hỏi phải biết lợi nhuận tháng 7. Vào tháng 5, ta chưa có thông tin về lợi nhuận tháng 7, nên không thể dự báo. Trong thiết lập này, dùng giá trị quá khứ là cách tốt hơn.

Đôi khi ta cũng phát hiện những giá trị đặc trưng đầu vào cực kỳ khác biệt so với phần lớn dữ liệu. Nếu lo ngại tác động của chúng lên hiệu suất thuật toán ML, ta có thể chọn một số phương pháp kinh nghiệm để xử lý. Các phương pháp này đặt ngưỡng để xác định một giá trị có phải ngoại lai hay không:

  * Bất kỳ điểm nào nằm ngoài khoảng $[\mu−m\sigma,\mu+m\sigma]$ có thể coi là ngoại lai. $\mu$ là trung bình mẫu và $\sigma$ là độ lệch chuẩn. Mức khắt khe khi gắn nhãn ngoại lai được điều chỉnh bằng bội số $m$, thường thuộc tập $\{3,5,10\}$.
  * Nếu giá trị lớn nhất lớn hơn $m$ lần giá trị lớn thứ nhì, nó cũng có thể được xếp vào ngoại lai (lập luận tương tự cho phía bên trái của phân phối).
  * Với một ngưỡng nhỏ $q$ cho trước, mọi giá trị nằm ngoài khoảng phân vị $[q,1−q]$ có thể coi là ngoại lai. Khoảng của $q$ thường là (0,5%; 5%), trong đó 1% và 2% được dùng nhiều nhất.

Sau khi đã gắn nhãn các giá trị ngoại lai, ta phải quyết định có đưa chúng vào dữ liệu hay không và đưa vào như thế nào. Nếu quyết định giữ lại, cách làm phổ biến nhất là thay giá trị ngoại lai bằng ngưỡng trên hoặc ngưỡng dưới tương ứng mà ta thu được theo một trong các phương pháp trên.

### **2.3 Chọn đặc trưng và chuẩn hóa thang đo**

Nếu có sẵn một tập lớn các biến dự báo, việc lọc bỏ các đặc trưng đầu vào dư thừa hoặc không cần thiết là hợp lý. Các phương pháp đơn giản gồm:

* Tính ma trận tương quan của mọi đặc trưng đầu vào và lọc các biến sao cho không có giá trị (tuyệt đối) nào vượt một ngưỡng (chẳng hạn 0,7), để các biến dư thừa không làm nhiễu bộ máy học.
* Huấn luyện các hồi quy tuyến tính và loại bỏ các biến không có ý nghĩa thống kê.
* Dùng các phương pháp không giám sát (phân cụm hoặc phân tích thành phần chính) để giữ lại số lượng đặc trưng rút gọn làm đầu vào cho thuật toán.

Các phương pháp trên có thể bỏ sót các quan hệ phi tuyến. Một cách tiếp cận khác là khớp một cây quyết định (hoặc rừng ngẫu nhiên) và chỉ giữ lại các đặc trưng có tầm quan trọng biến cao, hoặc khai thác kiến trúc autoencoder. Bạn sẽ học các phương pháp này ở những module và khóa học khác.

Cuối cùng, như đã nhấn mạnh ở bài trước, nhiều ứng dụng ML cần co giãn thang đo. Các thuật toán tối ưu sẽ học tốt hơn về mức độ liên quan của từng đặc trưng đầu vào nếu chúng có chung thang đo. Trong phân tích dữ liệu ở trên, ta đã thực hiện hai phiên bản co giãn đặc trưng: chuẩn hóa z-score và co giãn min-max về khoảng đơn vị. Với `scikitlearn`, bên dưới ta lại dùng co giãn min-max, nhưng trong khoảng $[-1,1]$, để huấn luyện mô hình.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_input = MinMaxScaler(feature_range=(-1, 1))
scaler_input.fit(X_train)
X_train = scaler_input.transform(X_train)
X_test = scaler_input.transform(X_test)

## **3. Mô hình logistic**

Bây giờ ta huấn luyện mô hình dự báo. Ta không chỉ định tham số cụ thể nào khi định nghĩa lớp `logisticRegr`, nghĩa là dùng các tùy chọn tối ưu mặc định. Điều này có nghĩa mô hình được điều chuẩn hóa bằng Ridge, với tham số $\alpha$ đặt bằng 1. Hãy xem tài liệu hướng dẫn để hiểu các tùy chọn khác nhau dùng để tinh chỉnh việc huấn luyện mô hình.

Bên dưới là các tham số đã huấn luyện của mô hình, cho ta biết tác động của từng biến lên xác suất vỡ nợ. Ví dụ, các hộ có tỷ lệ nợ trên thu nhập cao hơn có xác suất vỡ nợ lớn hơn, điều này dễ hiểu về mặt kinh tế. Tuy nhiên, một số tham số khác thoạt nhìn có thể kém hợp lý. Chẳng hạn, người lớn tuổi hơn, thâm niên lâu hơn hay có bằng đại học dường như liên quan dương với vỡ nợ. Lưu ý rằng tập dữ liệu này cho biết hành vi vỡ nợ của cá nhân *có điều kiện* là cá nhân đó đã nhận được khoản vay. Ví dụ, người lớn tuổi thường có mức nợ thấp hơn người trẻ. Do đó, những người lớn tuổi đang có dư nợ nhiều khả năng có tình trạng tài chính xấu hơn so với những người trẻ khá giống họ. Ta có thể diễn giải tương tự cho các tham số còn lại.

Dù sao, nhiệm vụ cốt lõi ở đây là xây dựng một mô hình dự báo tốt ngoài mẫu, chứ không phải lý giải mối quan hệ giữa từng đầu vào cụ thể và xác suất vỡ nợ. Do đó, các tham số đã huấn luyện ít quan trọng hơn nếu ta thu được một mô hình dự báo tốt.

In [ ]:
from sklearn.linear_model import LogisticRegression

# parameters not specified, all are set to their defaults
logisticRegr = LogisticRegression()
logisticRegr.fit(X_train, y_train)

In [ ]:
print(logisticRegr.intercept_)
for cc in range(len(X.keys())):
    print(X.keys()[cc], logisticRegr.coef_[0, cc])

Bên dưới là độ chính xác của mô hình đã huấn luyện, khoảng 92,5%. Xác suất vỡ nợ trung bình trong mẫu là 11,3%, nên mô hình dường như phân biệt được phần nào các trường hợp vỡ nợ trong mẫu kiểm tra.

In [ ]:
predictions = logisticRegr.predict(X_test)

In [ ]:
# Use score method to get accuracy of model
score = logisticRegr.score(X_test, y_test)
print(score)

Vì chỉ phân tích độ chính xác của mô hình có thể gây hiểu lầm, ta sẽ lấy thêm các thước đo hiệu suất đã mô tả ở bài trước.

Trước hết, ta lấy và vẽ ma trận nhầm lẫn. Lúc này hãy để ý rằng mô hình thất bại khá nặng khi dự báo nhiều sự kiện vỡ nợ. Nghĩa là mô hình khá thận trọng và trong phần lớn trường hợp dự báo cá nhân sẽ không vỡ nợ. Điều này cho ra ít "dương tính giả" nhưng rất nhiều "âm tính giả".

In [ ]:
from sklearn import metrics

cm = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(9, 9))
sns.heatmap(cm, annot=True, fmt=".3f", linewidths=0.5, square=True, cmap="Blues_r")
plt.ylabel("Actual label")
plt.xlabel("Predicted label")
all_sample_title = "Accuracy Score: {0}".format(score)
plt.title(all_sample_title, size=15);

Precision của nhóm 1 trên tập kiểm tra cho biết trong tất cả các dự báo vỡ nợ của mô hình, 98% thực sự là những người xin vay "xấu". Recall của nhóm 1 trên tập kiểm tra cho biết tỷ lệ người xin vay vỡ nợ mà mô hình nhận diện được là người xin vay "xấu". Như vậy, mô hình chỉ nhận diện được 34% số người xin vay "xấu".

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

Bên dưới là đường cong ROC, cho biết hiệu quả phân loại của mô hình khi ta muốn tăng recall, đổi lại tỷ lệ dương tính giả tăng lên. Diện tích dưới đường cong (AUC) thấp hơn 1 khá nhiều. Lưu ý rằng có thể đạt 0% dương tính giả bằng cách chỉ chấp nhận khoảng 40% dương tính thật, gần với kết quả của mô hình đã huấn luyện ở ngưỡng xác suất 50%. Muốn đạt tỷ lệ dương tính thật cao hơn chỉ có thể bằng cách trả giá là dự báo nhiều cá nhân "tốt" thành xấu hơn. Do đó, với một mô hình chưa hoàn hảo như thế này, ta có thể tùy ý chọn mô hình thiên về đánh giá cao hay đánh giá thấp xác suất vỡ nợ.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

sns.set(style="whitegrid", color_codes=True)
logit_roc_auc = roc_auc_score(y_test, logisticRegr.predict(X_test))
fpr, tpr, thresholds = roc_curve(y_test, logisticRegr.predict_proba(X_test)[:, 1])
plt.figure()
plt.plot(fpr, tpr, label="Logistic Regression (area = %0.2f)" % logit_roc_auc)
plt.plot([0, 1], [0, 1], "r--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver operating characteristic")
plt.legend(loc="lower right")
plt.show()

Một cách đơn giản để cải thiện dự báo vỡ nợ của mô hình là tăng độ phức tạp bằng lũy thừa của các đặc trưng đầu vào, như ta đã làm ở Bài 1. Thêm vào như vậy, ta có thể cải thiện độ khớp của mô hình nhờ tính đến các quan hệ phi tuyến giữa vỡ nợ và các đặc trưng đầu vào. Ví dụ, xác suất vỡ nợ có thể không tăng hay giảm đơn điệu theo tuổi; xác suất có thể thấp hơn ở người trung niên so với người trẻ hoặc người lớn tuổi.

Trong đoạn mã dưới đây, ta huấn luyện mô hình hồi quy logistic có đầu vào gồm lũy thừa đến bậc 6 của mỗi đặc trưng đầu vào. Khi tối ưu, ta tăng số vòng lặp tối đa so với giá trị mặc định là 100 vì mô hình phức tạp hơn. Ngoài ra, ta cũng điều chỉnh tham số điều chuẩn hóa, đặt "C" bằng 2,5. Tham số này là chỉ báo nghịch đảo của cường độ điều chuẩn hóa, tức tham số $\alpha$ ở Bài 2, nghĩa là ta điều chuẩn hóa nhẹ hơn trường hợp mặc định có giá trị 1. Bạn có thể xem tài liệu của hàm và thử các tham số của mô hình để xem hiệu suất trong các thiết lập khác nhau.

Kết quả bên dưới cho thấy việc thêm lũy thừa của các đặc trưng đầu vào mang lại cải thiện đáng kể. Recall tăng lên 54%, vẫn khá tệ nhưng tốt hơn nhiều so với mô hình trước. Điều này cho thấy các quan hệ phi tuyến có vai trò quan trọng để cải thiện dự báo của mô hình.

Rất khó đánh giá tầm quan trọng của các quan hệ phi tuyến này bằng cách quan sát dữ liệu trực tiếp. Các công cụ khác trong học máy, như mạng nơ-ron, có thể "nhận diện" chúng một cách linh hoạt mà không cần áp đặt chặt chẽ dạng hàm, và cải thiện khả năng học của mô hình. Bạn sẽ học các chủ đề này ở những module sau.

In [ ]:
import numpy as np

# Redefine the input feature matrix to include powers of each feature
Xpoly = X
for pp in range(2, 6):
    Xpoly = np.concatenate((Xpoly, np.power(X, pp)), axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    Xpoly, y, test_size=int(len(y) * 0.5), shuffle=False
)
# Scale the features
scaler_input = MinMaxScaler(feature_range=(-1, 1))
scaler_input.fit(X_train)
X_train = scaler_input.transform(X_train)
X_test = scaler_input.transform(X_test)
# Set up Logistic Regression
logisticRegr = LogisticRegression(C=2.5, max_iter=500)
logisticRegr.fit(X_train, y_train)
# Display coefficients
print(logisticRegr.intercept_)
print(logisticRegr.coef_)
# Compute accuracy
predictions = logisticRegr.predict(X_test)
score = logisticRegr.score(X_test, y_test)
print(score)
# Display confusion matrix and other indicators
cm = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(9, 9))
sns.heatmap(cm, annot=True, fmt=".3f", linewidths=0.5, square=True, cmap="Blues_r")
plt.ylabel("Actual label")
plt.xlabel("Predicted label")
all_sample_title = "Accuracy Score: {0}".format(score)
plt.title(all_sample_title, size=15)
print(classification_report(y_test, predictions))

In [ ]:
logit_roc_auc = roc_auc_score(y_test, logisticRegr.predict(X_test))
fpr, tpr, thresholds = roc_curve(y_test, logisticRegr.predict_proba(X_test)[:, 1])
plt.figure()
plt.plot(fpr, tpr, label="Logistic Regression (area = %0.2f)" % logit_roc_auc)
plt.plot([0, 1], [0, 1], "r--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver operating characteristic")
plt.legend(loc="lower right")
plt.show()

## **4. Kết luận**

Ta đã đi qua một ca nghiên cứu hoàn chỉnh về dự báo xác suất vỡ nợ bằng mô hình logistic. Ở các module tiếp theo, bạn sẽ học thêm các kỹ thuật học máy hữu ích khác.